# Assignment — LangChain Fundamentals
## Landscape through Tools

**Domain for this assignment: GreenPlate — a restaurant reservation and food ordering
assistant.** This is a deliberately different scenario from CineBot, used throughout your
notebooks — the goal is to prove you understand the *concepts*, not that you can copy-paste
code you've already seen with new variable names.

**Structure:** Part A is conceptual (no coding), Part B is coding exercises, both arranged from
easier to harder. Part C is a single capstone challenge that combines everything. Attempt
sections in order — later questions build on ideas from earlier ones.

**Before you start:** make sure your environment is set up (API key loaded) and you can run a
basic `model.invoke()` successfully.

**A note on collaboration:** discussing concepts with classmates is encouraged. Copying code
without understanding it will be obvious the moment a follow-up question asks you to modify or
explain it.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("FIREWORKS_API_KEY"), "Missing FIREWORKS_API_KEY -- check your .env file or Colab Secrets"

from langchain.chat_models import init_chat_model
model = init_chat_model(
    model_provider="fireworks",
    model="accounts/fireworks/models/kimi-k3",
    api_key=os.getenv("FIREWORKS_API_KEY"),
    base_url="https://api.fireworks.ai/inference/v1"
)
print("Environment ready.")

# This assignment uses LangChain v1-style APIs in the exercises below.
# Recommended packages if you need to install/update them:
# pip install -U langchain langchain-core langgraph langchain-fireworks pydantic python-dotenv


---
# Part A — Conceptual Questions

## A1. Foundations (Easy)

### 1. Agent loop vs. harness
An **agent** is the core decision loop: the model receives the current conversation/state, decides whether to answer or call a tool, observes the tool result, and repeats until the task is complete. The **harness** is everything that makes that loop useful, controlled, and production-ready: system prompts, tool definitions, middleware, runtime context, short-term and long-term memory, state management, structured-output handling, retries/error handling, guardrails, subagents/skills, persistence/checkpointing, tracing/observability, and the surrounding application/runtime.

### 2. The four products in the Lang family
- **LangChain** — the high-level framework for building applications and agents around models, tools, middleware, retrieval, structured output, and other components.
- **LangGraph** — the lower-level orchestration/runtime layer for stateful agent workflows, graphs, persistence, interrupts, and durable execution; LangChain agents run on LangGraph underneath.
- **LangSmith** — the observability, evaluation, tracing, testing, and deployment platform used to inspect and improve LLM/agent applications.
- **Deep Agents** — a higher-level agent harness built for longer, more complex tasks, adding capabilities such as planning, file-system use, context management, and subagents.

**LangSmith is fundamentally different in kind** because it is mainly an observability/evaluation/platform product around applications, while LangChain, LangGraph, and Deep Agents are primarily used to construct or run agentic applications.

### 3. `AgentExecutor` / `initialize_agent` in a 2026 tutorial
I would treat the tutorial as using an older LangChain API, even if the article itself was published recently. In LangChain v1, I should prefer `langchain.agents.create_agent()` for new agents and use middleware/runtime features around that API rather than starting new code with `AgentExecutor` or `initialize_agent`.

### 4. Why use a `.env` file?
A `.env` file keeps secrets such as API keys outside normal source code. `python-dotenv` can load those values into environment variables, while the `.env` file can be excluded from Git with `.gitignore`. If I hardcode a key inside a notebook cell, it can leak through Git history, notebook sharing, screenshots, outputs, backups, or code review; rotating the key later also becomes harder because the secret is mixed with code. A `.env` file is not magic security by itself—it must still be protected and not committed—but it separates configuration/secrets from source code.

### 5. Plain text, message objects, and dictionary messages
- A **plain text prompt** such as `model.invoke("Describe butter chicken")` is simplest for a one-turn request where I only need a user message.
- A **message-object list** such as `[SystemMessage(...), HumanMessage(...)]` is natural when I want explicit LangChain message types, richer metadata, tool messages, or programmatic conversation manipulation.
- A **dictionary-based message list** such as `[{"role": "system", "content": ...}, {"role": "user", "content": ...}]` is a concise, serializable form that is convenient for APIs, JSON payloads, stored chat histories, and `create_agent().invoke()` calls.

## A2. Models, Messages, and Templates (Medium)

### 6. Useful `AIMessage` fields besides `.content`
At least four important attributes are:
- **`tool_calls`** — structured tool requests produced by the model, including tool name, arguments, and call ID.
- **`usage_metadata`** — token usage information such as input, output, and total tokens; useful for cost/usage monitoring.
- **`response_metadata`** — provider-specific response details such as finish reason, model/provider metadata, or other execution information.
- **`id`** — a unique message identifier useful for tracing and correlating messages.
- **`content_blocks`** — a standardized view of typed blocks such as text, reasoning, citations, or multimodal content.
- **`invalid_tool_calls`** (when present) — malformed tool calls that could not be parsed correctly, useful for debugging tool-call failures.

### 7. Why streaming returns `AIMessageChunk`
Streaming is not just splitting a finished string into pieces. A model can stream text **and structured metadata**, including partial tool-call arguments, IDs, usage information, reasoning/content blocks, and provider metadata. `AIMessageChunk` preserves those semantics. Most importantly, compatible chunks are **additive/composable**: LangChain can combine them with `+` to reconstruct a meaningful full message, including assembled tool calls, rather than merely concatenating text characters.

### 8. `.batch()` vs `.batch_as_completed()`
`.batch()` runs multiple inputs concurrently but returns the final results in the same order as the input list after the batch is resolved. `.batch_as_completed()` yields `(index, result)` pairs as individual calls finish, so faster calls can be processed immediately even if earlier inputs are still running.

I would prefer `.batch_as_completed()` for something like generating descriptions for 200 menu items where I want to display/save each result as soon as it arrives, handle individual failures quickly, or start downstream work without waiting for the slowest model call.

### 9. `ToolMessage.content` vs `ToolMessage.artifact`
`content` is the tool result that is placed back into the model's conversation context, so it should contain the compact information the model needs. `artifact` is extra programmatic data attached to the tool result that is **not sent to the model**.

A RAG tool benefits from this separation because it can put the short retrieved passage/summary in `content`, while keeping full `Document` objects, document IDs, page numbers, scores, URLs, or other metadata in `artifact` for citation rendering or downstream application logic without wasting model context tokens.

### 10. Literal JSON inside `ChatPromptTemplate`
Prompt templates use braces for variables. If I paste `{"status": "ok"}` directly into a Python-format template, LangChain may interpret the braces as a template variable/expression and formatting can fail. I should escape literal braces by doubling them:

```python
'Example: {{"status": "ok"}}'
```

Then `{dish_name}` can still be used normally for a real template variable.

### 11. What `MessagesPlaceholder` does
`MessagesPlaceholder` reserves a position in a `ChatPromptTemplate` for **zero or more already-structured messages**. At formatting time I can inject a conversation history such as `HumanMessage`, `AIMessage`, and `ToolMessage` objects while preserving their roles, metadata, and boundaries.

A normal string variable would flatten that history into one piece of text. It would lose native message roles and structured message information, so it is not equivalent when the model needs an actual sequence of chat messages.

## A3. Structured Output (Medium-Hard)

### 12. Why “please respond in JSON” is less reliable
A plain prompt instruction is only a natural-language request. The model can still add prose, omit a key, use the wrong type, invent a different field name, or output syntactically invalid JSON. `with_structured_output()` gives LangChain a real schema and uses the model/provider's structured-output or tool-calling mechanism, then parses the result; with a Pydantic schema it also validates types and constraints. That turns formatting from a suggestion into a machine-checkable contract.

### 13. What `model.profile` is and how it affects structured output
`model.profile` is capability metadata attached to a chat model. Depending on the integration it can describe things such as context-window size, supported input/output modalities, tool calling, reasoning support, and native structured-output support.

The important distinction is that **profile-driven automatic strategy selection belongs to agent-level structured output**: when `create_agent(..., response_format=MySchema)` receives a bare schema, LangChain can inspect profile capability data and choose provider-native structured output when supported, otherwise a tool-calling strategy. At the raw model level, `model.with_structured_output(MySchema)` delegates to that model integration's implementation/default `method`; I do not pass `ProviderStrategy`/`ToolStrategy` objects to this method.

### 14. Code review of `with_structured_output(..., strategy=ProviderStrategy(...))`
This mixes the **agent-level** strategy API with the **model-level** method. `with_structured_output()` expects the schema directly (plus provider-specific options such as `method` where supported), not `strategy=ProviderStrategy(...)`.

Correct model-level usage:

```python
structured_model = model.with_structured_output(BookingRequest)
```

If I specifically want agent-level provider strategy, that belongs on `create_agent`:

```python
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(BookingRequest),
)
```

### 15. Model-level vs agent-level structured output
`model.with_structured_output(Schema)` wraps one model call so that call returns parsed data matching the schema. It is ideal for extraction/classification tasks that do not need an agent loop.

`create_agent(..., response_format=...)` makes structured output part of the **agent's final-result contract**. The agent can call ordinary tools first, observe results, retry after schema errors, continue looping, and finally place the validated object in `result["structured_response"]`. That is why agent-focused work uses `response_format`: a raw structured model call by itself does not orchestrate the multi-step tool loop.

### 16. Ambiguous message with `ToolStrategy(Union[...])`
LangChain exposes the possible structured schemas as tool-like choices and the model chooses the one that best fits the request. If the request is genuinely ambiguous, the model may still pick one based on context. If it incorrectly produces **multiple structured-output calls**, `ToolStrategy`'s default error handling sends error feedback in a `ToolMessage` and asks the model to retry with the single most relevant schema. If the application cannot tolerate semantic ambiguity, I should add clearer system instructions or explicitly ask the user for clarification rather than assuming schema validation alone can infer intent.

### 17. Validation failure for `party_size=50`
1. The user asks for a table for 50.
2. The model attempts to produce structured output with `party_size=50`.
3. The Pydantic schema validates `party_size: int = Field(ge=1, le=20)` and rejects `50`.
4. With `ToolStrategy(..., handle_errors=True)` (the default), the agent converts that validation failure into a `ToolMessage` containing feedback about the invalid structured output.
5. That error message is appended to the agent loop and the model gets another turn.
6. The model must retry with a schema-valid result, **but validation cannot magically make the user's real request satisfiable**. A good system prompt should tell the model what business rule to apply—for example, cap online reservations at 20 and explain that larger parties require private-dining assistance, or return a different schema/status representing that the request cannot be accepted.
7. Once the model produces data satisfying the schema and business policy, LangChain validates it and returns it as `structured_response`.

The key point is that schema self-correction fixes the **shape/constraint violation**; the application still needs a policy for semantically impossible requests.

## A4. Tools (Hard)

### 18. Is the docstring “the tool's entire pitch to the model”?
For ordinary `@tool` functions, the claim is largely correct: the model sees the tool name, input schema, and description, and the function docstring becomes the default description. A clear docstring tells the model **when** the tool is appropriate and what it does, so vague descriptions can reduce tool-selection accuracy.

There are cases where the docstring matters less—for example, when I explicitly supply a high-quality `description=` to the tool decorator, or when orchestration deterministically forces a tool instead of asking the model to choose. But in normal agentic selection the description/docstring is part of the model-visible interface, so it is much more than human documentation.

### 19. What `ToolRuntime` hides, and how
`ToolRuntime` gives the Python tool access to runtime-only information such as agent state, invocation context, the long-term store, stream writer, execution information/config, and the current tool-call ID. The model should not have to invent or supply these internal values.

LangChain knows to hide it because the tool function parameter is **type-annotated as `ToolRuntime`** (for example, `runtime: ToolRuntime`). That parameter is injected by the runtime and omitted from the JSON/tool schema shown to the model.

### 20. `runtime.state` vs `runtime.context` vs `runtime.store`
| Runtime value | Lifetime | Example |
|---|---|---|
| `runtime.state` | Mutable short-term state for the current conversation/thread; with a checkpointer it can persist across invocations of that same thread. | Conversation messages, a temporary cart, or a per-thread counter. |
| `runtime.context` | Immutable per-invocation dependency/configuration supplied when calling the agent. It is not conversational memory. | Restaurant location, authenticated user ID, membership tier, database client. |
| `runtime.store` | Long-term persistent key/value memory designed to survive across conversations/threads when backed by a persistent store. | Dietary preference, favorite dish, user profile settings. |

### 21. Accidentally declaring a parameter named `config`
`config` is a reserved tool parameter name used internally for LangChain's `RunnableConfig`. If I expose my own ordinary tool argument named `config`, I can get runtime/schema/injection errors because LangChain interprets that name specially rather than as a normal model-supplied argument.

It is easy to do accidentally because `config` is an extremely natural parameter name in regular Python functions. For agent tools I should use a domain-specific name such as `order_options` or access execution configuration through the supported runtime/config mechanism instead.

### 22. Plain string return vs `Command`
A plain string is an **observation**: it is returned to the model as the tool result, but by itself it does not directly update arbitrary fields in agent state. A `Command` can perform graph/state updates (and can also include an appropriate `ToolMessage`), so it is used when the tool must change the running workflow's state.

Original example: a `select_shipping_address` tool should set `selected_address_id` in agent state so every later tax, inventory, and shipping tool reads the same selected address. Returning only `"Address A selected"` would make the model aware of the selection, but would not reliably mutate the canonical `selected_address_id` state field. Returning `Command(update={...})` makes that state transition explicit and machine-controlled.

### 23. Why invisible tool gating is stronger than a prompt rule
A system prompt saying “do not use the premium tool for standard users” is a behavioral instruction to the model; models can misunderstand instructions, be manipulated by conflicting text, or simply make mistakes. With `wrap_model_call`, I can remove the restricted tool from `request.tools` before the model call. The model then receives **no callable schema for that tool**, so it cannot legitimately select it in that turn. This is an enforcement boundary rather than a request for compliance.

### 24. What is a headless tool?
A **headless tool** has a model-visible name/description/schema on the server but no server-side implementation. When the model calls it, agent execution is interrupted and the tool call is sent to the client application; the browser/desktop/mobile client executes the implementation locally, sends the result back, and the agent resumes.

That is fundamentally different from the normal tools in this assignment, whose Python implementation executes where the agent/server runs. A realistic headless-only capability is reading the user's browser geolocation through `navigator.geolocation` (with browser permission): the server cannot directly execute that browser API on the user's device, while a client-side headless tool can.


---
# Part B — Coding Exercises

All exercises use **GreenPlate**, a restaurant reservation and food ordering assistant. Each
question gives you a blank code cell to work in — write your solution there and run it to
confirm it works before moving on.

## B1. Easy

**B1.1 — A basic tool.** Write a `@tool`-decorated function called `check_table_availability`
that takes a `party_size: int` and a `time_slot: str`, and returns a string saying whether a
table is available (you can hardcode fake availability data, e.g. tables available for parties
of 2-6 at "7:00 PM" and "8:30 PM" only). Include a proper docstring. Print the tool's `.name`,
`.description`, and `.args` to confirm it's built correctly.


In [ ]:
from langchain.tools import tool

@tool
def check_table_availability(party_size: int, time_slot: str) -> str:
    """Check whether GreenPlate has a table for the requested party size and time slot."""
    available_slots = {"7:00 PM", "8:30 PM"}
    valid_party_size = 2 <= party_size <= 6

    if valid_party_size and time_slot in available_slots:
        return f"Available: table for {party_size} at {time_slot}."
    return f"Unavailable: no table for {party_size} at {time_slot}."

print("Name:", check_table_availability.name)
print("Description:", check_table_availability.description)
print("Args:", check_table_availability.args)


**B1.2 — A reusable prompt template.** Build a `ChatPromptTemplate` that generates a short,
enthusiastic description of a dish, given `{dish_name}` and `{cuisine_type}` as variables. Run
it with at least two different dish/cuisine combinations and print both results.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# The template is reusable because only the dish and cuisine values change.
dish_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are GreenPlate's menu writer. Keep descriptions to 1-2 enthusiastic sentences."),
    ("human", "Describe the {cuisine_type} dish {dish_name}."),
])

dish_chain = dish_prompt | model

for values in [
    {"dish_name": "Butter Chicken", "cuisine_type": "Indian"},
    {"dish_name": "Margherita Pizza", "cuisine_type": "Italian"},
]:
    response = dish_chain.invoke(values)
    print(f"{values['dish_name']}: {response.content}\n")


**B1.3 — A schema with a constrained field.** Define a Pydantic `BaseModel` called
`FoodOrder` with fields: `customer_name` (str), `dish_name` (str), `quantity` (int, must be
between 1 and 10), and `spice_level` (a `Literal` restricted to `"mild"`, `"medium"`, or
`"hot"`). Use `with_structured_output()` to extract a `FoodOrder` from this message: *"Hi, I'm
Karan, 2 butter chicken please, medium spice."* Print the result.


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class FoodOrder(BaseModel):
    """A validated GreenPlate food order."""
    customer_name: str = Field(description="Name of the customer")
    dish_name: str = Field(description="Dish the customer wants to order")
    quantity: int = Field(ge=1, le=10, description="Number of dishes, from 1 through 10")
    spice_level: Literal["mild", "medium", "hot"] = Field(description="Requested spice level")

structured_food_model = model.with_structured_output(FoodOrder)

food_order = structured_food_model.invoke(
    "Hi, I'm Karan, 2 butter chicken please, medium spice."
)
print(food_order)


## B2. Medium

**B2.1 — Two schemas, one agent.** A restaurant assistant needs to handle both new reservations
and cancellations. Define `NewReservation` (customer_name, party_size, time_slot) and
`CancelReservation` (customer_name, time_slot) as separate Pydantic models. Build a
`create_agent` with `response_format=ToolStrategy(Union[NewReservation, CancelReservation])`,
and test it with one clearly-a-reservation message and one clearly-a-cancellation message. Use
`isinstance()` to print which schema was chosen each time.


In [ ]:
from typing import Union
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class NewReservation(BaseModel):
    """A request to create a new restaurant reservation."""
    customer_name: str = Field(description="Customer name")
    party_size: int = Field(ge=1, le=20, description="Number of guests")
    time_slot: str = Field(description="Requested reservation time")

class CancelReservation(BaseModel):
    """A request to cancel an existing restaurant reservation."""
    customer_name: str = Field(description="Customer name")
    time_slot: str = Field(description="Reservation time to cancel")

reservation_agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[NewReservation, CancelReservation]),
    system_prompt=(
        "Classify the user's intent into exactly one of the provided reservation schemas. "
        "Do not invent extra reservation operations."
    ),
)

create_result = reservation_agent.invoke({
    "messages": [{"role": "user", "content": "I'm Priya. Reserve a table for 4 at 7:00 PM."}]
})
cancel_result = reservation_agent.invoke({
    "messages": [{"role": "user", "content": "I'm Omar. Cancel my 8:30 PM reservation."}]
})

for label, result in [("create", create_result), ("cancel", cancel_result)]:
    value = result["structured_response"]
    if isinstance(value, NewReservation):
        chosen = "NewReservation"
    elif isinstance(value, CancelReservation):
        chosen = "CancelReservation"
    else:
        chosen = type(value).__name__
    print(label, "->", chosen, value)


**B2.2 — A tool with `args_schema`.** Define a Pydantic input schema called `OrderInput` with
`dish_name` (str, with a description), `quantity` (int, `ge=1, le=10`, with a description), and
`delivery_or_pickup` (a `Literal["delivery", "pickup"]`, defaulting to `"pickup"`). Build a tool
called `place_order` using `args_schema=OrderInput`. Print `place_order.args` to confirm the
schema came through correctly, including the constraint and the default.


In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

class OrderInput(BaseModel):
    """Input accepted by the GreenPlate ordering tool."""
    dish_name: str = Field(description="Name of the dish to order")
    quantity: int = Field(ge=1, le=10, description="Quantity to order, between 1 and 10")
    delivery_or_pickup: Literal["delivery", "pickup"] = Field(
        default="pickup",
        description="Whether the order is for delivery or pickup",
    )

@tool(args_schema=OrderInput)
def place_order(dish_name: str, quantity: int, delivery_or_pickup: str = "pickup") -> str:
    """Place a GreenPlate food order after all order details are known."""
    return f"Placed {quantity} x {dish_name} for {delivery_or_pickup}."

print(place_order.args)


**B2.3 — Deliberately trigger a validation failure and watch self-correction.** Using the
`FoodOrder` schema from B1.3, build a `create_agent` with
`response_format=ToolStrategy(FoodOrder)`. Send a message asking for 15 units of a dish (this
should violate your `quantity` constraint). Print the full `result["messages"]` trace and point
out, in a comment, exactly where the self-correction happens.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

validation_agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(FoodOrder),
    system_prompt=(
        "Extract a FoodOrder. The maximum allowed quantity is 10. "
        "If the customer requests more than 10, retry using quantity=10 so the result satisfies the schema."
    ),
)

result = validation_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi, I'm Karan. I want 15 butter chicken, hot spice."
    }]
})

for i, message in enumerate(result["messages"]):
    print(f"\n--- message {i}: {type(message).__name__} ---")
    print(message)

print("\nFinal structured response:", result["structured_response"])

# SELF-CORRECTION: inspect the ToolMessage after the model's first invalid structured-output
# attempt. ToolStrategy reports the Pydantic validation error (quantity > 10) to the model.
# The next model turn retries with a schema-valid quantity, which is then accepted.


## B3. Hard

**B3.1 — Long-term memory with `ToolRuntime`.** Build two tools: `save_dietary_preference`
(takes `customer_id`, `preference`, and `runtime: ToolRuntime`, saves to `runtime.store`) and
`recall_dietary_preference` (takes `customer_id` and `runtime: ToolRuntime`, reads it back).
Build an agent with these two tools and a real `store=` attached. Prove the memory survives
across two **separate** `.invoke()` calls — save a preference in the first call, and recall it
correctly in a second, independent call.


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore

@tool
def save_dietary_preference(
    customer_id: str,
    preference: str,
    runtime: ToolRuntime,
) -> str:
    """Save a customer's dietary preference for future conversations."""
    if runtime.store is None:
        return "No long-term store is configured."
    runtime.store.put(
        ("dietary_preferences",),
        customer_id,
        {"preference": preference},
    )
    return f"Saved dietary preference for {customer_id}: {preference}."

@tool
def recall_dietary_preference(customer_id: str, runtime: ToolRuntime) -> str:
    """Recall a customer's previously saved dietary preference."""
    if runtime.store is None:
        return "No long-term store is configured."
    item = runtime.store.get(("dietary_preferences",), customer_id)
    if item is None:
        return f"No dietary preference saved for {customer_id}."
    return f"Saved dietary preference for {customer_id}: {item.value['preference']}."

preference_store = InMemoryStore()

memory_agent = create_agent(
    model=model,
    tools=[save_dietary_preference, recall_dietary_preference],
    store=preference_store,
    system_prompt=(
        "You are GreenPlate. When the user asks to remember a dietary preference, call "
        "save_dietary_preference. When asked what was remembered, call "
        "recall_dietary_preference. Always use the supplied customer_id exactly."
    ),
)

# Invocation 1: write long-term memory.
first = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My customer_id is CUST-101. Remember that I prefer vegetarian food."
    }]
})
print("FIRST INVOCATION:")
print(first["messages"][-1].content)

# Invocation 2: completely separate message history, same agent/store.
second = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My customer_id is CUST-101. What dietary preference did I ask you to remember?"
    }]
})
print("\nSECOND INVOCATION:")
print(second["messages"][-1].content)

# Direct proof that the value lives in the store, not merely in message history.
print("\nSTORE VALUE:", preference_store.get(("dietary_preferences",), "CUST-101").value)


**B3.2 — Dynamic tool gating.** GreenPlate has a `book_private_dining_room` tool that should
only be available to customers with a "premium" membership tier. Using `wrap_model_call`, write
a middleware function that removes this tool from the model's visible toolset unless
`request.state.get("is_premium_member")` is `True`. Prove it works by running the SAME query
twice — once without the flag, once with it — and show the tool is genuinely unavailable in the
first case (not just "declined").


In [ ]:
from collections.abc import Callable
from typing_extensions import NotRequired
from langchain.tools import tool
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@tool
def book_private_dining_room(event_name: str, party_size: int) -> str:
    """Book GreenPlate's private dining room for an eligible premium member."""
    return f"Private dining room booked for {event_name}, party of {party_size}."

@tool
def explain_membership() -> str:
    """Explain that private dining booking is a premium-member feature."""
    return "Private dining room booking is available to premium members."

class PremiumState(AgentState):
    is_premium_member: NotRequired[bool]

@wrap_model_call(state_schema=PremiumState)
def gate_private_dining(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Hide the private-room tool unless the current state marks the user as premium."""
    is_premium = request.state.get("is_premium_member", False)
    visible_tools = list(request.tools)

    if not is_premium:
        visible_tools = [
            t for t in visible_tools
            if t.name != "book_private_dining_room"
        ]

    # This print is our proof of what the MODEL can actually see on this turn.
    print("Model-visible tools:", [t.name for t in visible_tools])
    return handler(request.override(tools=visible_tools))

premium_agent = create_agent(
    model=model,
    tools=[book_private_dining_room, explain_membership],
    middleware=[gate_private_dining],
    system_prompt=(
        "Use book_private_dining_room when it is available and the user requests a private room. "
        "If that tool is unavailable, explain the premium requirement."
    ),
)

query = {
    "messages": [{
        "role": "user",
        "content": "Book the private dining room for my Anniversary Dinner for 12 people."
    }]
}

print("NON-PREMIUM RUN")
non_premium = premium_agent.invoke({**query, "is_premium_member": False})
for m in non_premium["messages"]:
    print(type(m).__name__, getattr(m, "tool_calls", None), m.content)

print("\nPREMIUM RUN")
premium = premium_agent.invoke({**query, "is_premium_member": True})
for m in premium["messages"]:
    print(type(m).__name__, getattr(m, "tool_calls", None), m.content)

# In the first run, the middleware's "Model-visible tools" printout omits
# book_private_dining_room entirely. Therefore the model cannot call it; this is
# stronger than merely telling the model to decline the request.


**B3.3 — Combine structured output and tools in one agent.** Build a `create_agent` that has
BOTH a `check_table_availability`-style tool AND a `response_format=ReservationConfirmation`
schema (define this schema yourself — it should capture at minimum: customer_name, time_slot,
confirmed: bool). Send a request that requires the agent to actually call the tool to check
availability *before* it can correctly fill in `confirmed`. Print both `result["messages"]` and
`result["structured_response"]`, and explain in a comment why this required the agent-level
`response_format`, not the raw model-level `with_structured_output()`.


In [ ]:
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

@tool
def lookup_table_availability(party_size: int, time_slot: str) -> str:
    """Check actual GreenPlate table availability before confirming a reservation."""
    available_slots = {"7:00 PM", "8:30 PM"}
    if 2 <= party_size <= 6 and time_slot in available_slots:
        return "AVAILABLE"
    return "UNAVAILABLE"

class ReservationConfirmation(BaseModel):
    """Final result after GreenPlate checks table availability."""
    customer_name: str = Field(description="Customer name")
    time_slot: str = Field(description="Requested reservation time")
    confirmed: bool = Field(description="True only when the availability tool returned AVAILABLE")

confirmation_agent = create_agent(
    model=model,
    tools=[lookup_table_availability],
    response_format=ToolStrategy(ReservationConfirmation),
    system_prompt=(
        "You are GreenPlate's reservation agent. You MUST call lookup_table_availability before "
        "setting confirmed. Set confirmed=True only when that tool returns AVAILABLE; otherwise False."
    ),
)

result = confirmation_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'm Elena. Please reserve a table for 4 at 7:00 PM."
    }]
})

print("MESSAGE TRACE")
for i, message in enumerate(result["messages"]):
    print(f"\n--- {i}: {type(message).__name__} ---")
    print(message)

print("\nSTRUCTURED RESPONSE")
print(result["structured_response"])

# This needs agent-level response_format because the correct final value depends on a
# preceding tool call. create_agent can loop: model -> availability tool -> model ->
# validated ReservationConfirmation. A raw model.with_structured_output() call only
# structures that model invocation; it does not itself orchestrate this tool loop.


---
# Part C — Capstone Challenge

Build a single, complete `create_agent` for GreenPlate that combines **at least five** of the
following in one working system (your choice which five, but justify your choices in a markdown
cell before your code):

- A structured schema for orders or reservations (with at least one real constraint, like a
  `Literal` or a numeric range)
- At least two custom tools
- Long-term memory via `ToolRuntime.store` (e.g. remembering a customer's dietary preferences or
  favorite dish across sessions)
- Short-term memory via a checkpointer and `thread_id` (e.g. remembering the customer's name
  within one conversation)
- Dynamic tool gating based on some condition (membership tier, time of day, order size — your
  choice)
- A `context_schema` carrying some per-run data a tool reads (e.g. `restaurant_location`)

**Requirements:**
1. A markdown cell explaining your design choices before the code.
2. The complete, runnable code.
3. At least two `.invoke()` calls that demonstrate the system actually working — not just that
   it builds without error.
4. A short markdown reflection (3-5 sentences) on ONE trade-off or limitation of your design —
   what would break, or what would you need to add, if this went to real production use.

This is intentionally open-ended. There is no single correct architecture — the goal is
demonstrating you can combine these pieces into something coherent, not matching a hidden answer
key.


*Design explanation (before the code):*

The capstone combines **six** LangChain features in one GreenPlate agent:

1. **Structured output** — `GreenPlateResult` gives the application a predictable final object. `quantity` is constrained to 1-10 and `action`/`fulfillment` use `Literal` values.
2. **Multiple custom tools** — tools save/recall dietary preferences, inspect the restaurant location, place an order, and optionally book a private room.
3. **Long-term memory** — `ToolRuntime.store` keeps dietary preferences by `customer_id`, so they can survive outside one conversation thread (a real application would replace `InMemoryStore` with persistent storage).
4. **Short-term memory** — `InMemorySaver` plus a `thread_id` preserves message/state history across calls in the same conversation. This lets the second call refer back to the customer's name from the first call.
5. **Dynamic tool gating** — middleware removes `book_private_dining_room` from the model-visible tools unless the per-run membership tier is `premium`.
6. **Runtime context** — `GreenPlateContext` supplies `customer_id`, `restaurant_location`, and `membership_tier` without asking the model to generate those trusted application values.

I chose these features because they separate responsibilities cleanly: conversational history belongs in the checkpointer, durable user preferences belong in the store, trusted request metadata belongs in context, authorization is enforced by middleware, tools perform actions, and structured output gives the caller a stable final contract.


In [ ]:
from collections.abc import Callable
from dataclasses import dataclass
from typing import Literal

from pydantic import BaseModel, Field
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


# ---------- Trusted per-run context ----------
@dataclass
class GreenPlateContext:
    customer_id: str
    restaurant_location: str
    membership_tier: Literal["standard", "premium"] = "standard"


# ---------- Final structured response ----------
class GreenPlateResult(BaseModel):
    """Validated final result returned by the GreenPlate agent."""
    action: Literal["preference_saved", "preference_recalled", "order", "private_room", "information"]
    customer_name: str | None = Field(default=None, description="Customer name when known")
    dish_name: str | None = Field(default=None, description="Dish for an order")
    quantity: int | None = Field(default=None, ge=1, le=10, description="Order quantity from 1 to 10")
    fulfillment: Literal["pickup", "delivery"] | None = None
    status: Literal["success", "unavailable", "needs_information"]
    message: str = Field(description="Short user-facing outcome")


# ---------- Long-term-memory tools ----------
@tool
def save_preference(preference: str, runtime: ToolRuntime[GreenPlateContext]) -> str:
    """Save the current customer's dietary preference for future conversations."""
    if runtime.store is None:
        return "ERROR: long-term store is unavailable."
    customer_id = runtime.context.customer_id
    runtime.store.put(
        ("greenplate", "dietary_preferences"),
        customer_id,
        {"preference": preference},
    )
    return f"Saved preference for {customer_id}: {preference}."


@tool
def recall_preference(runtime: ToolRuntime[GreenPlateContext]) -> str:
    """Recall the current customer's saved dietary preference before recommending or ordering food."""
    if runtime.store is None:
        return "No preference store is available."
    customer_id = runtime.context.customer_id
    item = runtime.store.get(("greenplate", "dietary_preferences"), customer_id)
    if item is None:
        return "No dietary preference is saved for this customer."
    return f"Saved dietary preference: {item.value['preference']}."


# ---------- Context-aware and action tools ----------
@tool
def get_restaurant_location(runtime: ToolRuntime[GreenPlateContext]) -> str:
    """Get the GreenPlate restaurant location selected for this run."""
    return runtime.context.restaurant_location


@tool
def submit_food_order(
    dish_name: str,
    quantity: int,
    fulfillment: Literal["pickup", "delivery"] = "pickup",
) -> str:
    """Submit a GreenPlate food order after required details and dietary constraints are known."""
    if not 1 <= quantity <= 10:
        return "ORDER_REJECTED: quantity must be between 1 and 10."
    return f"ORDER_CONFIRMED: {quantity} x {dish_name}, {fulfillment}."


@tool
def book_private_dining_room(event_name: str, party_size: int) -> str:
    """Book GreenPlate's private dining room. This capability is for premium members only."""
    if party_size < 2 or party_size > 30:
        return "PRIVATE_ROOM_UNAVAILABLE: supported party size is 2-30."
    return f"PRIVATE_ROOM_CONFIRMED: {event_name}, party of {party_size}."


# ---------- Dynamic authorization/gating middleware ----------
@wrap_model_call
def premium_tool_gate(
    request: ModelRequest[GreenPlateContext],
    handler: Callable[[ModelRequest[GreenPlateContext]], ModelResponse],
) -> ModelResponse:
    """Make the private-room tool invisible to non-premium customers."""
    tier = request.runtime.context.membership_tier
    tools = list(request.tools)
    if tier != "premium":
        tools = [t for t in tools if t.name != "book_private_dining_room"]

    print(f"Membership={tier}; model-visible tools={[t.name for t in tools]}")
    return handler(request.override(tools=tools))


# ---------- Agent infrastructure ----------
long_term_store = InMemoryStore()
short_term_checkpointer = InMemorySaver()

capstone_agent = create_agent(
    model=model,
    tools=[
        save_preference,
        recall_preference,
        get_restaurant_location,
        submit_food_order,
        book_private_dining_room,
    ],
    response_format=ToolStrategy(GreenPlateResult),
    context_schema=GreenPlateContext,
    store=long_term_store,
    checkpointer=short_term_checkpointer,
    middleware=[premium_tool_gate],
    system_prompt=(
        "You are GreenPlate, a restaurant ordering assistant. "
        "Use tools instead of guessing. When asked to remember a dietary preference, call save_preference. "
        "Before ordering when a saved preference may matter, call recall_preference. "
        "Call get_restaurant_location when the user asks where pickup will be. "
        "Call submit_food_order to actually place an order. "
        "If book_private_dining_room is visible and requested, use it. If it is not visible, explain that "
        "private-room booking requires premium membership. "
        "After tool work is complete, always return exactly one GreenPlateResult."
    ),
)


# ---------- Invocation 1: save preference + establish short-term conversation context ----------
thread_config = {"configurable": {"thread_id": "greenplate-demo-1"}}
standard_context = GreenPlateContext(
    customer_id="CUST-501",
    restaurant_location="GreenPlate Midtown Atlanta",
    membership_tier="standard",
)

result1 = capstone_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Hi, I'm Maya. Remember that I am vegan."
        }]
    },
    config=thread_config,
    context=standard_context,
)
print("\nINVOCATION 1 STRUCTURED RESPONSE")
print(result1["structured_response"])


# ---------- Invocation 2: same thread, name is not repeated; recall memory + place order ----------
result2 = capstone_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Use my saved dietary preference, order 2 tofu bowls for pickup, "
                "and tell me which restaurant location the pickup is from."
            )
        }]
    },
    config=thread_config,
    context=standard_context,
)
print("\nINVOCATION 2 STRUCTURED RESPONSE")
print(result2["structured_response"])

print("\nLONG-TERM STORE PROOF")
print(long_term_store.get(("greenplate", "dietary_preferences"), "CUST-501").value)


# ---------- Invocation 3: demonstrate dynamic gating with a premium run ----------
premium_context = GreenPlateContext(
    customer_id="CUST-777",
    restaurant_location="GreenPlate Midtown Atlanta",
    membership_tier="premium",
)
result3 = capstone_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "I'm Daniel. Book the private dining room for a team dinner of 14 people."
        }]
    },
    config={"configurable": {"thread_id": "greenplate-demo-premium"}},
    context=premium_context,
)
print("\nINVOCATION 3 STRUCTURED RESPONSE")
print(result3["structured_response"])


*Reflection on trade-offs/limitations:*

This design uses `InMemoryStore` and `InMemorySaver`, which are excellent for learning but lose data when the Python process ends, so a production deployment would need durable, shared persistence. The tools also use hardcoded fake restaurant behavior; real ordering and reservation tools would need database/API integrations, idempotency keys, authentication, retries, and transactional error handling. Dynamic tool hiding is a useful authorization layer, but production authorization should still be enforced again inside the protected service/tool so security does not depend on model visibility alone. Finally, the structured schema validates output shape and constraints, but business-level correctness still requires deterministic server-side validation before charging a customer or committing a reservation.
